# 01 · Vector search: the baseline and its gap

Look at the data, then run the simplest retrieval there is, and watch where it fails.

| | |
| --- | --- |
| **Needs** | `00_setup.ipynb` in the same folder. The first code cell runs it for you. |
| **LLM** | Optional. With `RUN_LLM = False` (the default in `00_setup.ipynb`) you still see everything retrieval does. |
| **Previous / next** | [`00_setup.ipynb`](00_setup.ipynb) / [`02_bm25_ner.ipynb`](02_bm25_ner.ipynb) |

*The code mirrors the [`rag-retrieval` journey](https://github.com/instaclustr/labs/tree/main/rag-retrieval) in `instaclustr/labs`. Lines marked "Mirrors" name the file in the journey that each piece comes from. This notebook only searches: it never writes to the cluster.*

### Run the setup

The next cell runs `00_setup.ipynb`. It connects to the cluster, loads the two small models, and defines the helper functions used below. You should see `[PASS]` lines. If you see `[FAIL]`, fix that before going on, using the notes in `00_setup.ipynb`.

In [ ]:
%run ./00_setup.ipynb

## The data

The cluster holds **260 recipes**, stored as **1,823 chunks**. A *chunk* is one section of a recipe: its ingredients, its allergen cautions and labels, its introduction, its method (three steps per chunk), or its serving notes. Every search in these notebooks returns chunks, and the code folds them back into recipes, so you read the results one recipe at a time.

Two things about the chunks are worth knowing:

- **Each chunk says which recipe it belongs to.** Its text starts with the recipe name and source, for example `Fish and chips (source: womensweeklyfood.com.au) - Allergen cautions and labels: ...`. Without that prefix, a chunk such as "Method steps 4-6" would not say which recipe it came from.
- **Chunks follow the recipe's own sections.** There is no fixed chunk size and no overlap between chunks. Every chunk also carries its recipe's metadata: `cautions` (allergen warnings such as Sulfites) and a `tier`.

The tier is a governance label that notebook 05 explains: recipes with any caution are `hot`, the rest are `lt` (long-term). The ingredients, cautions and labels come from the original recipe records. The introduction, method and serving text is **synthetic**: a language model wrote it from each recipe's ingredient list, for this demo.

The cell below asks the cluster to count them. It is an aggregation, which is a read-only search. A recipe has many chunks, so it counts distinct `recipe_id` values to count recipes.

In [ ]:
recipes = {"cardinality": {"field": "recipe_id"}}   # distinct recipes, not chunks
body = {
    "size": 0, "track_total_hits": True,
    "aggs": {
        "recipes": recipes,
        "by_kind": {"filters": {"filters": {
            "ingredients": {"prefix": {"section": "Ingredients"}},
            "allergen cautions": {"prefix": {"section": "Allergen"}},
            "introduction": {"prefix": {"section": "Introduction"}},
            "method steps": {"prefix": {"section": "Method"}},
            "serving and storage": {"prefix": {"section": "Serving"}},
        }}},
        "by_tier": {"terms": {"field": "tier", "size": 5}, "aggs": {"recipes": recipes}},
        "sulfites": {"filter": {"term": {"cautions": "sulfites"}}, "aggs": {"recipes": recipes}},
        "top_cautions": {"terms": {"field": "cautions", "size": 6}, "aggs": {"recipes": recipes}},
    },
}
r = search(IDX["vector"], body)
a = r["aggregations"]
print(f"chunks: {r['hits']['total']['value']}   recipes: {a['recipes']['value']}")
print("chunks by kind:", {k: v["doc_count"] for k, v in a["by_kind"]["buckets"].items()})
print("recipes by tier:", {b["key"]: b["recipes"]["value"] for b in a["by_tier"]["buckets"]})
print("recipes with a Sulfites caution:", a["sulfites"]["recipes"]["value"])
print("most common cautions (recipes):", {b["key"]: b["recipes"]["value"] for b in a["top_cautions"]["buckets"]})

**Expected:** 1,823 chunks that make up 260 recipes, about 7 chunks per recipe. The kinds add up to the chunk total, with the method split into several chunks per recipe. 201 recipes have a Sulfites caution, and the tier split is 209 `hot` and 51 `lt`.

### The two "Fish and Chips" recipes

The rest of the notebook turns on two recipes that share a name but come from different websites. They differ in one safety-relevant fact: one lists a Sulfites caution and the other does not. A good retriever must tell them apart.

In [ ]:
body = {"size": chunk_fetch(10), "_source": RECIPE_FIELDS,
        "query": {"match_phrase": {"recipe_name": "fish and chips"}}}
pair = group_by_recipe(normalize_hits(search(IDX["vector"], body)), top_k=10)
show_hits(pair)

**Expected:** the two recipes named Fish and Chips, one from `thegratefulgirlcooks.com` with **no** cautions (tier `lt`), and one from `womensweeklyfood.com.au` with a **Sulfites** caution (tier `hot`). Other fish-and-chips-style recipes may also appear.

Keep these two in mind. Every step below asks the same two questions:

- `Q_GRATEFUL`: *Does the thegratefulgirlcooks.com Fish and Chips recipe contain sulfites?* (Answer: no.)
- `Q_WOMENS`: *Does the womensweeklyfood.com.au Fish and Chips recipe contain sulfites?* (Answer: yes.)

## Baseline vector RAG

**The idea.** Turn the question into a vector and find the chunks whose vectors are nearest. This is *semantic* search: it matches on meaning, not on exact words. The search asks for six chunks for every recipe wanted, because several chunks of one recipe often rank together, and then folds the chunks into recipes (each recipe keeps its best chunk's score).

**The failure to look for.** Both questions mean almost the same thing and mention the same dish, so their vectors are almost identical. The recipe's *website name* barely moves the vector. Watch whether the recipe each question names actually shows up.

*Mirrors:* `scripts/baseline-vector-rag/src/query.py`.

In [ ]:
def vector_search(question, top_k=5, candidates=25, index=None):
    n_chunks = chunk_fetch(top_k)   # ask for more chunks than recipes: one recipe can fill several slots
    body = {"size": n_chunks, "_source": RECIPE_FIELDS,
            "query": {"knn": {"embedding": {"vector": embed(question), "k": max(candidates, n_chunks)}}}}
    return group_by_recipe(normalize_hits(search(index or IDX["vector_baseline"], body)), top_k=top_k)


def ask_vector(question, top_k=5):
    hits = vector_search(question, top_k)
    prompt = f"Question:\n{question}\n\nContext:\n{build_context(hits) or 'No context available.'}"
    return answer(prompt, VECTOR_SYSTEM), hits


for question, domain in [(Q_GRATEFUL, "thegratefulgirlcooks"), (Q_WOMENS, "womensweeklyfood")]:
    reply, hits = ask_vector(question)
    print("\nQ:", question)
    show_hits(hits)
    print(f"  -> the named recipe ({domain}) was retrieved: {has_source(hits, domain)}")
    print("  ANSWER:", reply)

**Expected:** 5 recipes per question. Under each one, `matched:` lists the sections of the chunks that matched. For most of them it is only "Allergen cautions and labels", because the question is mostly "does it contain sulfites?", which matches the allergen chunk of nearly every recipe. For the first question, the `thegratefulgirlcooks.com` recipe is **not** retrieved, so the system cannot answer about the right recipe. For the second, the `womensweeklyfood.com.au` recipe is retrieved. Scores and answer wording can vary. This is the **retrieval gap**: vector search found *similar* recipes, not the *named* one.

**Try this:** change `top_k` to 10 and rerun. Does the missing recipe appear? (It does not. Even with 10 recipes it stays out.)

### Checkpoint

You should be able to say why vector search did not retrieve the `thegratefulgirlcooks.com` recipe for a question that names it. (Both questions mean almost the same thing, so their vectors are almost identical, and the website name barely moves a vector.)

Next: open [`02_bm25_ner.ipynb`](02_bm25_ner.ipynb).